# Factorised DQN on Atari

This notebook is self-contained: it uses only Gymnasium, ALE, NumPy, PyTorch, and Matplotlib. It applies the repository's factorisation principle to a shared visual encoder: `Q(s, a | task) = phi(s, a)^T psi(task)`. The same network is trained sequentially on three Atari games. Observations are four stacked 84x84 grayscale frames and actions are padded to ALE's 18-action vocabulary, so every task has compatible inputs and outputs.

Install the optional Atari runtime once if needed: `pip install "ale-py>=0.10.2"`.


### Mathematical Factorisation

The Q-function is factorised into a dot product between a state-action representation and a task representation:
$$ Q(s, a \mid z) = \phi(s, a)^\top \psi(z) $$

where:
* $s$ is the state (e.g., stacked grayscale frames).
* $a$ is the chosen action.
* $z$ is the task indicator (e.g., a one-hot vector).
* $\phi(s, a)$ is the state-action embedding produced by a shared encoder (a convolutional network for vision).
* $\psi(z)$ is the task embedding produced by the goal encoder.

This factorisation allows the visual encoder to be shared across multiple tasks while retaining task-specific capabilities. By aligning the representations, it enforces generalisation across the state-action space.


In [ ]:
import sys
from pathlib import Path

# Walk up to find the `src/` directory and add it to sys.path
_nb_dir = Path('.').resolve()
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / 'src').is_dir():
        _src = str(_p / 'src')
        if _src not in sys.path:
            sys.path.insert(0, _src)
        break

from utils import set_seed, evaluate_policy
from visualisations import print_goal_embedding_similarity

import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

try:
    import ale_py

    if hasattr(gym, "register_envs"):
        gym.register_envs(ale_py)
except ImportError as exc:
    raise ImportError(
        'Install the Atari runtime with: pip install "ale-py>=0.10.2"'
    ) from exc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TASKS = ("Breakout", "Pong", "SpaceInvaders")
MAX_ACTIONS = 18  # ALE's fixed legal-action upper bound.
FRAME_STACK = 4





def make_env(task, seed=0, render_mode=None):
    """Return a Gymnasium Atari environment with factorisation-compatible pixels."""
    env = gym.make(
        f"ALE/{task}-v5",
        frameskip=1,
        repeat_action_probability=0.0,
        render_mode=render_mode,
    )
    env = gym.wrappers.AtariPreprocessing(
        env, frame_skip=4, screen_size=84, grayscale_obs=True, scale_obs=False
    )
    stack = getattr(gym.wrappers, "FrameStackObservation", None)
    env = (
        stack(env, stack_size=FRAME_STACK)
        if stack
        else gym.wrappers.FrameStack(env, FRAME_STACK)
    )
    env.reset(seed=seed)
    return env


def obs_array(observation):
    return np.asarray(observation, dtype=np.float32) / 255.0


class FactorisedAtariQ(nn.Module):
    """Q(s,a|z) = normalised phi(s,a) dot normalised psi(z)."""

    def __init__(self, task_dim, rep_dim=128):
        super().__init__()
        self.visual = nn.Sequential(
            nn.Conv2d(FRAME_STACK, 32, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.state = nn.Sequential(
            nn.LazyLinear(512), nn.ReLU(), nn.Linear(512, rep_dim)
        )
        self.action = nn.Embedding(MAX_ACTIONS, rep_dim)
        self.goal_encoder = nn.Sequential(
            nn.Linear(task_dim, 128), nn.ReLU(), nn.Linear(128, rep_dim)
        )

    def encode_state_action(self, obs, actions):
        phi = self.state(self.visual(obs)) + self.action(actions)
        return F.normalize(torch.tanh(phi), dim=-1)

    def encode_task(self, task):
        return F.normalize(torch.tanh(self.goal_encoder(task)), dim=-1)

    def forward(self, obs, actions, task):
        return (self.encode_state_action(obs, actions) * self.encode_task(task)).sum(
            -1, keepdim=True
        )

    def q_values(self, obs, task, action_count):
        batch = obs.shape[0]
        actions = torch.arange(MAX_ACTIONS, device=obs.device).repeat(batch)
        repeated_obs = obs.repeat_interleave(MAX_ACTIONS, dim=0)
        repeated_task = task.repeat_interleave(MAX_ACTIONS, dim=0)
        values = self(repeated_obs, actions, repeated_task).view(batch, MAX_ACTIONS)
        values[:, action_count:] = -1e9  # Never choose padding actions.
        return values


class Replay:
    def __init__(self, capacity):
        self.data = deque(maxlen=capacity)

    def add(self, transition):
        self.data.append(transition)

    def sample(self, batch_size):
        indices = np.random.choice(len(self.data), batch_size, replace=False)
        rows = [self.data[i] for i in indices]
        obs, action, reward, next_obs, done, task, action_count = zip(*rows)
        return (
            torch.as_tensor(np.stack(obs), device=DEVICE),
            torch.as_tensor(action, device=DEVICE),
            torch.as_tensor(reward, dtype=torch.float32, device=DEVICE).unsqueeze(1),
            torch.as_tensor(np.stack(next_obs), device=DEVICE),
            torch.as_tensor(done, dtype=torch.float32, device=DEVICE).unsqueeze(1),
            torch.as_tensor(np.stack(task), dtype=torch.float32, device=DEVICE),
            torch.as_tensor(action_count, device=DEVICE),
        )

    def __len__(self):
        return len(self.data)


def task_vector(index):
    vector = np.zeros(len(TASKS), dtype=np.float32)
    vector[index] = 1.0
    return vector


def evaluate(model, task, task_index, episodes=3, seed=10_000):
    env = make_env(task, seed=seed)
    z = torch.as_tensor(task_vector(task_index), device=DEVICE).unsqueeze(0)

    def policy_fn(obs):
        obs_t = torch.as_tensor(obs_array(obs), device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            action = model.q_values(obs_t, z, env.action_space.n).argmax(1).item()
        return action

    mean_return, _ = evaluate_policy(env, policy_fn, episodes=episodes)
    env.close()
    return mean_return


def train_task(
    model,
    target,
    task,
    task_index,
    *,
    total_steps=100_000,
    warmup_steps=20_000,
    batch_size=32,
    replay_capacity=100_000,
    eval_every=10_000,
    seed=42,
):
    """Train one sequential task while retaining the factorised visual/task representations."""
    set_seed(seed + task_index)
    env = make_env(task, seed=seed + task_index)
    replay = Replay(replay_capacity)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    z_np = task_vector(task_index)
    z = torch.as_tensor(z_np, device=DEVICE).unsqueeze(0)
    obs, _ = env.reset(seed=seed + task_index)
    history = []
    for step in range(1, total_steps + 1):
        epsilon = max(0.05, 1.0 - 0.95 * step / 250_000)
        if step < warmup_steps or np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                obs_t = torch.as_tensor(obs_array(obs), device=DEVICE).unsqueeze(0)
                action = model.q_values(obs_t, z, env.action_space.n).argmax(1).item()
        next_obs, reward, terminated, truncated, _ = env.step(action)
        replay.add(
            (
                obs_array(obs),
                action,
                reward,
                obs_array(next_obs),
                terminated or truncated,
                z_np,
                env.action_space.n,
            )
        )
        obs = next_obs
        if terminated or truncated:
            obs, _ = env.reset()
        if len(replay) >= max(warmup_steps, batch_size):
            states, actions, rewards, next_states, dones, tasks, counts = replay.sample(
                batch_size
            )
            if not torch.all(counts == counts[0]):
                raise RuntimeError(
                    "Internal replay consistency check failed: one task must use one action-space size."
                )
            with torch.no_grad():
                next_q = (
                    target.q_values(next_states, tasks, int(counts[0]))
                    .max(1, keepdim=True)
                    .values
                )
                bellman = rewards + 0.99 * (1 - dones) * next_q
            prediction = model(states, actions, tasks)
            loss = F.smooth_l1_loss(prediction, bellman)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            optimizer.step()
            with torch.no_grad():
                for source, destination in zip(model.parameters(), target.parameters()):
                    destination.lerp_(source, 0.005)
        if step % eval_every == 0:
            score = evaluate(model, task, task_index)
            history.append((step, score))
            print(f"{task:14s} step={step:7d} return={score:7.2f}")
    env.close()
    return history


# A small default run verifies the complete experiment. Increase TOTAL_STEPS for research runs.
TOTAL_STEPS = 10_000
probe_env = make_env(TASKS[0])
sample_obs, _ = probe_env.reset()
probe_env.close()
q = FactorisedAtariQ(len(TASKS)).to(DEVICE)
with torch.no_grad():
    q(
        torch.as_tensor(obs_array(sample_obs), device=DEVICE).unsqueeze(0),
        torch.zeros(1, dtype=torch.long, device=DEVICE),
        torch.eye(len(TASKS), device=DEVICE)[:1],
    )
target = FactorisedAtariQ(len(TASKS)).to(DEVICE)
target.load_state_dict(q.state_dict())
results = {
    task: train_task(q, target, task, index, total_steps=TOTAL_STEPS)
    for index, task in enumerate(TASKS)
}
with torch.no_grad():
    task_embeddings = q.encode_task(torch.eye(len(TASKS), device=DEVICE)).cpu().numpy()
overall_results = {
    task: {"eval_returns": points, "task_embeddings": [task_embeddings[index]]}
    for index, (task, points) in enumerate(results.items())
}

fig, ax = plt.subplots(figsize=(8, 4))
for task, points in results.items():
    if points:
        ax.plot(*zip(*points), marker="o", label=task)
ax.set(
    xlabel="Environment steps within task",
    ylabel="Mean return",
    title="Sequential factorised Atari DQN",
)
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print_goal_embedding_similarity(task_embeddings, goal_labels=TASKS)
